In [19]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RANDOM_STATE, TEST_SIZE
from src.utils import evaluate_model

In [1]:
# Data Manipulation
import pandas as pd
import numpy as np

# Machine Learning Utilities
from sklearn.model_selection import train_test_split

# Preprocessing Tools
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

In [3]:
# Read dataset
df = pd.read_csv("../data/raw/ai4i2020.csv")

# Display first five rows
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
# Remove identifier columns
df = df.drop(columns=[
    "UDI",
    "Product ID"
])

# Remove failure subtype columns
# (Target leakage, known only after failure occures)
df = df.drop(columns=[
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
])

df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [5]:
#Feature engineering
# Temperature Difference
# Difference between process and ambient temperature. It represnts thermal load on the machine process

df["Temperature Difference [K]"] = (
    df["Process temperature [K]"] -
    df["Air temperature [K]"]
)

df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Temperature Difference [K]
0,M,298.1,308.6,1551,42.8,0,0,10.5
1,L,298.2,308.7,1408,46.3,3,0,10.5
2,L,298.1,308.5,1498,49.4,5,0,10.4
3,L,298.2,308.6,1433,39.5,7,0,10.4
4,L,298.2,308.7,1408,40.0,9,0,10.5


In [6]:
# Features (X)
X = df.drop(columns="Machine failure")

# Target (y)
y = df["Machine failure"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (10000, 7)
Target Shape: (10000,)


In [ ]:
# Split Dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

#Stratify of y means: Both train & test maintain the same failure percentage
print("Training Samples:", X_train.shape)
print("Testing Samples :", X_test.shape)

Training Samples: (8000, 7)
Testing Samples : (2000, 7)


In [8]:
# Numerical Features
numerical_features = X_train.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

# Categorical Features
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temperature Difference [K]']

Categorical Features:
['Type']


In [9]:
# Numerical Preprocessing Pipeline

numerical_pipeline = Pipeline(steps=[

    # Fill missing values using median.
    ("imputer", SimpleImputer(strategy="median")),

    # Scale numerical variables.
    ("scaler", StandardScaler())

])

In [10]:
# Categorical Preprocessing Pipeline

categorical_pipeline = Pipeline(steps=[

    # Replace missing category with most frequent value.
    ("imputer", SimpleImputer(strategy="most_frequent")),

    # Convert Type into binary columns.
    ("encoder", OrdinalEncoder(categories=[['L','M','H']]))

])

In [11]:
# Column Transformer
# Applies different preprocessing
# to numerical and categorical columns.

preprocessor = ColumnTransformer(
    transformers=[

        ("num",
         numerical_pipeline,
         numerical_features),

        ("cat",
         categorical_pipeline,
         categorical_features)

    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [13]:
X_train.head(5)

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Temperature Difference [K]
4058,M,302.0,310.9,1456,47.2,54,8.9
1221,M,297.0,308.3,1399,46.4,132,11.3
6895,M,301.0,311.6,1357,45.6,137,10.6
9863,L,298.9,309.8,1411,56.3,84,10.9
8711,L,297.1,308.5,1733,28.7,50,11.4


In [ ]:
from sklearn.linear_model import LogisticRegression

# Complete ML Pipeline

logistic_pipeline = Pipeline(steps=[

    ("preprocessor", preprocessor),

    ("model", LogisticRegression(
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))

])

#class_weight='balanced' => Machine Failures are rare
logistic_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the

In [16]:
# Train Logistic Pipeline

logistic_pipeline.fit(X_train, y_train)

print("Training Completed.")

Training Completed.


In [17]:
# Predictions

y_pred = logistic_pipeline.predict(X_test)

y_prob = logistic_pipeline.predict_proba(X_test)[:,1]

y_prob

array([0.97094317, 0.53099566, 0.67637784, ..., 0.16562494, 0.62435157,
       0.01892688])

In [18]:
from sklearn.metrics import accuracy_score
print(f"\nOverall Accuracy: {accuracy_score(y_test, y_pred):.2%}")


Overall Accuracy: 82.05%


In [70]:
tree_numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

tree_preprocessor = ColumnTransformer([
    ("num", tree_numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[

    ("preprocessor", tree_preprocessor),

    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE
    ))

])

In [72]:
# Train RandomForest Pipeline

rf_pipeline.fit(X_train, y_train)

print("Training Completed.")

Training Completed.


In [73]:
# Predictions

y_pred_rf = rf_pipeline.predict(X_test)
print(f"\nOverall Accuracy: {accuracy_score(y_test, y_pred_rf):.2%}")


Overall Accuracy: 98.85%


In [74]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay
)

y_pred_rf = rf_pipeline.predict(X_test)
y_prob_rf = rf_pipeline.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred_rf))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1932
           1       0.91      0.74      0.81        68

    accuracy                           0.99      2000
   macro avg       0.95      0.87      0.90      2000
weighted avg       0.99      0.99      0.99      2000

ROC-AUC: 0.9766928510534649


In [1]:
# Save Processed Dataset (Stage 2)

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer

# 1. Load Raw Dataset

DATA_PATH = "../data/raw/ai4i2020.csv"

df = pd.read_csv(DATA_PATH)

print("Original Dataset Shape:", df.shape)

# 2. Remove Unnecessary & Leakage Columns

columns_to_drop = [
    "UDI",
    "Product ID",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
]

df = df.drop(columns=columns_to_drop)

# 3. Feature Engineering

df["Temperature Difference [K]"] = (
    df["Process temperature [K]"] -
    df["Air temperature [K]"]
)

# 4. Separate Features and Target

X = df.drop(columns=["Machine failure"])
y = df["Machine failure"]

# 5. Train-Test Split (Stratified)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training Samples : {X_train.shape[0]}")
print(f"Testing Samples  : {X_test.shape[0]}")

# 6. Identify Feature Types

numerical_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Temperature Difference [K]"
]

categorical_features = ["Type"]

# 7. Build Preprocessing Pipeline


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[["L", "M", "H"]]))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

# 8. Fit on Training Data Only

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 9. Convert Back to DataFrames

feature_names = numerical_features + categorical_features

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

# Add target column back
train_processed = X_train_processed.copy()
train_processed["Machine failure"] = y_train.values

test_processed = X_test_processed.copy()
test_processed["Machine failure"] = y_test.values

# 10. Save Processed Datasets

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

train_processed.to_csv(
    output_dir / "train_processed.csv",
    index=False
)

test_processed.to_csv(
    output_dir / "test_processed.csv",
    index=False
)

# Optional: Save the complete processed dataset
processed_full = pd.concat(
    [train_processed, test_processed],
    ignore_index=True
)

processed_full.to_csv(
    output_dir / "ai4i_processed.csv",
    index=False
)

print("\nProcessed datasets saved successfully!")
print(output_dir.resolve())

Original Dataset Shape: (10000, 14)
Training Samples : 8000
Testing Samples  : 2000

Processed datasets saved successfully!
C:\Users\UJJAL\OneDrive\Desktop\ML Projects\Predictive-Maintenance-ML\data\processed
